In [ ]:
# PROJETO DE ZOONOSES
# TRATAMENTO DOS DADOS BRUTOS — SQLITE
# Versão 1.0

# - Preserva os registros brutos.
# - Trata os cinco conjuntos.
# - Identifica duplicidades de animais.
# - Recupera relacionamentos pelo ID técnico.
# - Registra problemas e correções.
# - Evita processar novamente os mesmos registros.
# - Preserva registros de origem manual.

import os
import json
import sqlite3
from datetime import date, datetime
from collections import Counter, defaultdict

# 1. CONFIGURAÇÕES

PASTA_PROJETO = "/content/PROJETO_ZOONOSES"
CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)

VERSAO_TRATAMENTO = "1.0"

MUNICIPIOS_VALIDOS = {
    "Feijó",
    "Cruzeiro do Sul",
    "Sena Madureira",
    "Rio Branco",
    "Tarauacá"
}

ESPECIES_VALIDAS = {"Cachorro", "Gato"}
SEXOS_VALIDOS = {"Macho", "Fêmea"}
STATUS_VALIDOS = {"Ativo", "Inativo"}

MOTIVOS_ACOLHIMENTO = {
    "Maus-tratos",
    "Abandono",
    "Acidente",
    "Doença",
    "Recolhimento preventivo",
    "Entrega voluntária",
    "Animal perdido",
    "Resgate de filhotes",
    "Outros"
}

ORIGENS_ACOLHIMENTO = {
    "Resgate pela equipe",
    "Denúncia da população",
    "Entrega pelo tutor",
    "Órgão público",
    "Clínica veterinária",
    "Recolhimento nas ruas"
}

VACINAS_VALIDAS = {
    "Antirrábica",
    "V8",
    "V10",
    "Polivalente felina",
    "Leucemia felina"
}

SITUACOES_VACINA = {
    "Aplicada",
    "Pendente",
    "Aguardando reforço"
}

SITUACOES_DIAGNOSTICO = {
    "Em tratamento",
    "Em acompanhamento",
    "Recuperado"
}

SITUACOES_OCORRENCIA = {
    "Em investigação",
    "Em acompanhamento",
    "Concluída"
}

TIPOS_EVENTO = {"Óbito", "Eutanásia"}

CONJUNTOS = {
    "animais",
    "vacinacoes",
    "diagnosticos",
    "maus_tratos",
    "obitos_eutanasias"
}


# 2. CONEXAO COM O BANCO LOCAL DO COLAB — SEM GOOGLE DRIVE

if not os.path.exists(CAMINHO_BANCO):
    raise FileNotFoundError(
        "Banco SQLite não encontrado. Verifique o caminho."
    )

conexao = sqlite3.connect(
    CAMINHO_BANCO,
    timeout=60
)

conexao.row_factory = sqlite3.Row
conexao.execute("PRAGMA foreign_keys = ON")
conexao.execute("PRAGMA busy_timeout = 60000")

cursor = conexao.cursor()


# 3. VERIFICAÇÃO DA ESTRUTURA

def tabelas_existentes():

    return {
        linha["name"]
        for linha in cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
        """).fetchall()
    }


def colunas_tabela(nome):

    return {
        linha["name"]
        for linha in cursor.execute(
            f"PRAGMA table_info({nome})"
        ).fetchall()
    }


tabelas_obrigatorias = {
    "registros_brutos",
    "animais",
    "vacinacoes",
    "diagnosticos",
    "maus_tratos",
    "obitos_eutanasias",
    "auditoria",
    "lotes"
}

faltantes = tabelas_obrigatorias - tabelas_existentes()

if faltantes:
    raise Exception(
        f"Tabelas obrigatórias ausentes: {sorted(faltantes)}"
    )


# 4. TABELAS DE CONTROLE E QUALIDADE

cursor.execute("""
CREATE TABLE IF NOT EXISTS controle_processamento (
    id_registro_bruto INTEGER PRIMARY KEY,
    id_lote INTEGER,
    conjunto TEXT NOT NULL,
    versao_tratamento TEXT NOT NULL,
    status TEXT NOT NULL,
    id_animal_final TEXT,
    observacao TEXT,
    data_processamento TEXT NOT NULL,
    FOREIGN KEY (id_registro_bruto)
        REFERENCES registros_brutos(id_registro)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS qualidade_dados (
    id_qualidade INTEGER PRIMARY KEY AUTOINCREMENT,
    id_registro_bruto INTEGER,
    id_lote INTEGER,
    conjunto TEXT NOT NULL,
    id_animal TEXT,
    campo TEXT,
    valor_original TEXT,
    valor_corrigido TEXT,
    tipo_problema TEXT NOT NULL,
    acao TEXT NOT NULL,
    data_identificacao TEXT NOT NULL
)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_qualidade_conjunto
ON qualidade_dados(conjunto)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_controle_lote
ON controle_processamento(id_lote)
""")

conexao.commit()


# 5. FUNÇÕES AUXILIARES

hoje = date.today()


def texto(valor):

    if valor is None:
        return ""

    return str(valor).strip()


def data_valida(valor):

    valor = texto(valor)

    if not valor:
        return None

    try:
        data = date.fromisoformat(valor)

        if data > hoje:
            return None

        return data.isoformat()

    except (ValueError, TypeError):
        return None


def registrar_problema(
    id_registro,
    id_lote,
    conjunto,
    id_animal,
    campo,
    original,
    corrigido,
    problema,
    acao
):

    cursor.execute("""
        INSERT INTO qualidade_dados (
            id_registro_bruto,
            id_lote,
            conjunto,
            id_animal,
            campo,
            valor_original,
            valor_corrigido,
            tipo_problema,
            acao,
            data_identificacao
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        id_registro,
        id_lote,
        conjunto,
        id_animal,
        campo,
        json.dumps(original, ensure_ascii=False, default=str),
        json.dumps(corrigido, ensure_ascii=False, default=str),
        problema,
        acao,
        datetime.now().isoformat(timespec="seconds")
    ))


def marcar_processado(
    registro,
    status,
    id_animal_final=None,
    observacao=""
):

    cursor.execute("""
        INSERT OR IGNORE INTO controle_processamento (
            id_registro_bruto,
            id_lote,
            conjunto,
            versao_tratamento,
            status,
            id_animal_final,
            observacao,
            data_processamento
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        registro["id_registro"],
        registro["id_lote"],
        registro["conjunto"],
        VERSAO_TRATAMENTO,
        status,
        id_animal_final,
        observacao,
        datetime.now().isoformat(timespec="seconds")
    ))


def inserir_dinamico(tabela, dados, ignorar_conflito=False):

    colunas = colunas_tabela(tabela)

    dados_validos = {
        campo: valor
        for campo, valor in dados.items()
        if campo in colunas
    }

    if not dados_validos:
        raise Exception(
            f"Nenhuma coluna compatível para inserir em {tabela}."
        )

    campos = list(dados_validos)

    comando = (
        f"INSERT {'OR IGNORE ' if ignorar_conflito else ''}"
        f"INTO {tabela} "
        f"({', '.join(campos)}) "
        f"VALUES ({', '.join(['?'] * len(campos))})"
    )

    cursor.execute(
        comando,
        [dados_validos[campo] for campo in campos]
    )


def valor_mais_frequente(candidatos, campo, validador):

    valores = []

    for registro in candidatos:

        valor = registro.get(campo)

        if validador(valor):
            valores.append(valor)

    if not valores:
        return None

    contagem = Counter(valores)

    maior_frequencia = max(contagem.values())

    empatados = [
        valor
        for valor, quantidade in contagem.items()
        if quantidade == maior_frequencia
    ]

    # Em caso de empate, mantém a primeira ocorrência válida.
    for registro in candidatos:

        valor = registro.get(campo)

        if valor in empatados:
            return valor

    return empatados[0]


def campo_textual_valido(valor):

    return bool(texto(valor))


def peso_valido(valor, especie):

    try:

        peso = float(str(valor).replace(",", "."))

        if especie == "Gato":
            return 1 <= peso <= 12

        if especie == "Cachorro":
            return 1 <= peso <= 60

        return False

    except (ValueError, TypeError):
        return False


def municipio_da_imagem(imagem):

    imagem = texto(imagem)

    for municipio in MUNICIPIOS_VALIDOS:

        prefixo = municipio.replace(" ", "_") + "_animal_"

        if imagem.startswith(prefixo):
            return municipio

    return None


def inserir_qualidade_por_meta(
    registro,
    id_lote,
    conjunto,
    id_animal,
    campo,
    original,
    corrigido,
    problema,
    acao
):

    meta = registro.get("_meta") or {}

    problemas = meta.get("problemas_introduzidos") or []

    registrar_problema(
        registro["_id_registro"],
        id_lote,
        conjunto,
        id_animal,
        campo,
        original,
        corrigido,
        problema,
        acao
    )


# 6. LEITURA DOS REGISTROS BRUTOS

brutos = cursor.execute("""
    SELECT
        id_registro,
        id_lote,
        conjunto,
        dados_json
    FROM registros_brutos
    ORDER BY id_registro
""").fetchall()

registros_por_conjunto = defaultdict(list)

for linha in brutos:

    conjunto = linha["conjunto"]

    if conjunto not in CONJUNTOS:
        continue

    try:

        dados = json.loads(linha["dados_json"])

        if not isinstance(dados, dict):
            raise ValueError("Registro JSON não é um objeto.")

        dados["_id_registro"] = linha["id_registro"]
        dados["_id_lote"] = linha["id_lote"]
        dados["_conjunto"] = conjunto

        registros_por_conjunto[conjunto].append(dados)

    except Exception as erro:

        registrar_problema(
            linha["id_registro"],
            linha["id_lote"],
            conjunto,
            None,
            "dados_json",
            linha["dados_json"],
            None,
            "json_invalido",
            f"Registro não processado: {erro}"
        )


# 7. AGRUPAMENTO E TRATAMENTO DOS ANIMAIS

grupos_animais = defaultdict(list)

for animal in registros_por_conjunto["animais"]:

    meta = animal.get("_meta") or {}

    id_origem = texto(meta.get("id_animal_origem"))

    if not id_origem:

        id_origem = texto(animal.get("id_animal"))

    if not id_origem:

        registrar_problema(
            animal["_id_registro"],
            animal["_id_lote"],
            "animais",
            None,
            "id_animal",
            animal.get("id_animal"),
            None,
            "identificador_irrecuperavel",
            "Registro sem identificador técnico e sem ID público."
        )

        marcar_processado(
            animal,
            "REJEITADO",
            observacao="Identificador do animal irrecuperável."
        )

        continue

    grupos_animais[id_origem].append(animal)


mapa_animais = {}
eventos_por_animal = defaultdict(list)
quantidade_animais_inseridos = 0
quantidade_duplicidades = 0

for id_origem, candidatos in grupos_animais.items():

    # Evita reprocessar um grupo já concluído.
    ids_brutos = [
        item["_id_registro"]
        for item in candidatos
    ]

    ja_processados = cursor.execute("""
        SELECT COUNT(*)
        FROM controle_processamento
        WHERE id_registro_bruto IN (
            SELECT id_registro
            FROM registros_brutos
            WHERE id_registro IN (
                {}
            )
        )
    """.format(",".join("?" for _ in ids_brutos)), ids_brutos).fetchone()[0]

    if ja_processados == len(ids_brutos):

        existente = cursor.execute("""
            SELECT id_animal
            FROM animais
            WHERE id_animal = ?
        """, (id_origem,)).fetchone()

        if existente:
            mapa_animais[id_origem] = existente["id_animal"]

        continue

    # Escolhe os valores mais frequentes e válidos.
    especie = valor_mais_frequente(
        candidatos,
        "especie",
        lambda v: v in ESPECIES_VALIDAS
    )

    if not especie:
        especie = "Não informado"

    nome = valor_mais_frequente(
        candidatos,
        "nome",
        campo_textual_valido
    )

    raca = valor_mais_frequente(
        candidatos,
        "raca",
        campo_textual_valido
    )

    sexo = valor_mais_frequente(
        candidatos,
        "sexo",
        lambda v: v in SEXOS_VALIDOS
    )

    municipio = valor_mais_frequente(
        candidatos,
        "municipio",
        lambda v: v in MUNICIPIOS_VALIDOS
    )

    if not municipio:

        imagens = [
            municipio_da_imagem(item.get("imagem"))
            for item in candidatos
        ]

        imagens_validas = [
            item for item in imagens if item
        ]

        municipio = (
            Counter(imagens_validas).most_common(1)[0][0]
            if imagens_validas else None
        )

    peso = valor_mais_frequente(
        candidatos,
        "peso_kg",
        lambda v: peso_valido(v, especie)
    )

    # Verifica divergências de peso entre duplicidades.
    pesos_validos = []

    for item in candidatos:

        if peso_valido(item.get("peso_kg"), especie):

            pesos_validos.append(
                float(str(item["peso_kg"]).replace(",", "."))
            )

    revisar_peso = ""

    if len(set(pesos_validos)) > 1:

        revisar_peso = "Verificar peso"

        quantidade_duplicidades += 1

    elif peso is None:

        revisar_peso = "Verificar peso"

    status = valor_mais_frequente(
        candidatos,
        "status",
        lambda v: v in STATUS_VALIDOS
    )

    situacao_vacinacao = valor_mais_frequente(
        candidatos,
        "situacao_vacinacao",
        lambda v: v in {"Em dia", "Pendente", "Parcial"}
    )

    data_acolhimento = valor_mais_frequente(
        candidatos,
        "data_acolhimento",
        lambda v: data_valida(v) is not None
    )

    motivo_acolhimento = valor_mais_frequente(
        candidatos,
        "motivo_acolhimento",
        lambda v: v in MOTIVOS_ACOLHIMENTO
    )

    origem_acolhimento = valor_mais_frequente(
        candidatos,
        "origem_acolhimento",
        lambda v: v in ORIGENS_ACOLHIMENTO
    )

    imagem = valor_mais_frequente(
        candidatos,
        "imagem",
        campo_textual_valido
    )

    vacinas_faltantes = valor_mais_frequente(
        candidatos,
        "vacinas_faltantes",
        lambda v: v is not None
    )

    # Usa o ID técnico como identificador definitivo.
    id_final = id_origem

    # Não sobrescreve um cadastro manual com o mesmo ID.
    conflito = cursor.execute("""
        SELECT origem
        FROM animais
        WHERE id_animal = ?
    """, (id_final,)).fetchone()

    if conflito and conflito["origem"] == "Manual":

        id_final = "SINT-" + id_origem

    mapa_animais[id_origem] = id_final

    # Se houver evento terminal, o animal não pode permanecer ativo.
    possui_evento = any(
        texto(evento.get("_meta", {}).get("id_animal_origem"))
        == id_origem
        for evento in registros_por_conjunto["obitos_eutanasias"]
    )

    if possui_evento:
        status = "Inativo"

    if not nome:
        nome = "Não informado"

    if not raca:
        raca = "Não informada"

    if not sexo:
        sexo = "Não informado"

    if not municipio:
        municipio = "Não informado"

    if not status:
        status = "Ativo"

    if not situacao_vacinacao:
        situacao_vacinacao = "Não informado"

    if not motivo_acolhimento:
        motivo_acolhimento = "Não informado"

    if not origem_acolhimento:
        origem_acolhimento = "Não informado"

    # Registra os problemas encontrados nos candidatos.
    for item in candidatos:

        meta = item.get("_meta") or {}

        problemas_meta = meta.get("problemas_introduzidos") or []

        if problemas_meta:

            for problema in problemas_meta:

                registrar_problema(
                    item["_id_registro"],
                    item["_id_lote"],
                    "animais",
                    id_final,
                    "registro",
                    problema,
                    "Tratado ou encaminhado para revisão",
                    problema,
                    "Problema identificado pela marcação do gerador."
                )

    # Verificações adicionais de duplicidade.
    if len(candidatos) > 1:

        tipos = []

        for item in candidatos:

            problemas_meta = (
                item.get("_meta", {}).get("problemas_introduzidos", [])
            )

            tipos.extend(problemas_meta)

        if "duplicidade_exata" in tipos:

            quantidade_duplicidades += 1

        if "duplicidade_parcial" in tipos:

            quantidade_duplicidades += 1

    # Registra valores inválidos ou ausentes.
    campos_verificados = [
        ("nome", nome),
        ("raca", raca),
        ("sexo", sexo),
        ("municipio", municipio),
        ("peso_kg", peso),
        ("status", status),
        ("situacao_vacinacao", situacao_vacinacao),
        ("data_acolhimento", data_acolhimento),
        ("motivo_acolhimento", motivo_acolhimento),
        ("origem_acolhimento", origem_acolhimento)
    ]

    for item in candidatos:

        for campo, corrigido in campos_verificados:

            original = item.get(campo)

            invalido = (
                original != corrigido
                and (
                    original is None
                    or original == ""
                    or campo in {
                        "peso_kg",
                        "municipio",
                        "sexo",
                        "status",
                        "situacao_vacinacao",
                        "data_acolhimento",
                        "motivo_acolhimento",
                        "origem_acolhimento"
                    }
                )
            )

            if invalido:

                registrar_problema(
                    item["_id_registro"],
                    item["_id_lote"],
                    "animais",
                    id_final,
                    campo,
                    original,
                    corrigido,
                    "valor_ausente_ou_invalido",
                    "Valor padronizado ou encaminhado para revisão."
                )

    dados_animal = {
        "id_animal": id_final,
        "nome": nome,
        "especie": especie,
        "raca": raca,
        "sexo": sexo,
        "municipio": municipio,
        "peso_kg": peso,
        "status": status,
        "vacinas_faltantes": texto(vacinas_faltantes),
        "situacao_vacinacao": situacao_vacinacao,
        "imagem": imagem,
        "revisao_peso": revisar_peso,
        "data_cadastro": datetime.now().isoformat(timespec="seconds"),
        "id_lote": candidatos[0]["_id_lote"],
        "origem": "Sintético",
        "data_acolhimento": data_acolhimento,
        "motivo_acolhimento": motivo_acolhimento,
        "origem_acolhimento": origem_acolhimento
    }

    inserir_dinamico(
        "animais",
        dados_animal,
        ignorar_conflito=True
    )

    if cursor.rowcount > 0:
        quantidade_animais_inseridos += 1

    # Marca todos os registros brutos que formaram esse animal.

def marcar_processado(
    registro,
    status,
    id_animal_final=None,
    observacao=""
):
    # Recupera os identificadores adicionados durante a leitura
    id_registro = registro.get(
        "_id_registro",
        registro.get("id_registro")
    )

    id_lote = registro.get(
        "_id_lote",
        registro.get("id_lote")
    )

    conjunto = registro.get(
        "_conjunto",
        registro.get("conjunto")
    )

    if id_registro is None or conjunto is None:
        raise ValueError(
            "Registro sem identificador ou conjunto. "
            f"Dados recebidos: {registro}"
        )

    cursor.execute("""
        INSERT OR IGNORE INTO controle_processamento (
            id_registro_bruto,
            id_lote,
            conjunto,
            versao_tratamento,
            status,
            id_animal_final,
            observacao,
            data_processamento
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        id_registro,
        id_lote,
        conjunto,
        VERSAO_TRATAMENTO,
        status,
        id_animal_final,
        observacao,
        datetime.now().isoformat(timespec="seconds")
    ))



# 8. FUNÇÃO PARA RESOLVER RELACIONAMENTOS

def resolver_animal(registro):

    meta = registro.get("_meta") or {}

    id_origem = texto(meta.get("id_animal_origem"))

    if not id_origem:
        id_origem = texto(registro.get("id_animal"))

    if not id_origem:
        return None

    return mapa_animais.get(id_origem, id_origem)


# 9. TRATAMENTO DAS VACINAÇÕES

quantidade_vacinacoes = 0

for registro in registros_por_conjunto["vacinacoes"]:

    id_bruto = registro["_id_registro"]

    ja_processado = cursor.execute("""
        SELECT 1
        FROM controle_processamento
        WHERE id_registro_bruto = ?
    """, (id_bruto,)).fetchone()

    if ja_processado:
        continue

    id_animal = resolver_animal(registro)

    if not id_animal:

        marcar_processado(
            registro,
            "REJEITADO",
            observacao="Não foi possível identificar o animal."
        )

        continue

    animal_existe = cursor.execute("""
        SELECT 1
        FROM animais
        WHERE id_animal = ?
    """, (id_animal,)).fetchone()

    if not animal_existe:

        marcar_processado(
            registro,
            "NAO_VINCULADO",
            id_animal,
            "Animal não encontrado na tabela definitiva."
        )

        continue

    nome_vacina = registro.get("nome_vacina")

    if nome_vacina not in VACINAS_VALIDAS:

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "nome_vacina",
            nome_vacina,
            "Não informado",
            "vacina_ausente_ou_invalida",
            "Nome da vacina não pôde ser recuperado."
        )

        nome_vacina = "Não informado"

    data_aplicacao = data_valida(
        registro.get("data_aplicacao")
    )

    if not data_aplicacao:

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "data_aplicacao",
            registro.get("data_aplicacao"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    proxima_dose = data_valida(
        registro.get("proxima_dose")
    )

    if (
        proxima_dose
        and data_aplicacao
        and proxima_dose < data_aplicacao
    ):

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "proxima_dose",
            proxima_dose,
            None,
            "proxima_dose_anterior",
            "Data removida para revisão."
        )

        proxima_dose = None

    situacao = registro.get("situacao")

    if situacao not in SITUACOES_VACINA:
        situacao = "Pendente"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "situacao",
            registro.get("situacao"),
            situacao,
            "situacao_invalida",
            "Situação padronizada."
        )

    inserir_dinamico(
        "vacinacoes",
        {
            "id_animal": id_animal,
            "nome_vacina": nome_vacina,
            "data_aplicacao": data_aplicacao,
            "proxima_dose": proxima_dose,
            "situacao": situacao,
            "observacoes": texto(registro.get("observacoes"))
        }
    )

    marcar_processado(
        registro,
        "TRATADO",
        id_animal
    )

    quantidade_vacinacoes += 1


# 10. TRATAMENTO DOS DIAGNÓSTICOS

quantidade_diagnosticos = 0

for registro in registros_por_conjunto["diagnosticos"]:

    id_bruto = registro["_id_registro"]

    if cursor.execute("""
        SELECT 1
        FROM controle_processamento
        WHERE id_registro_bruto = ?
    """, (id_bruto,)).fetchone():
        continue

    id_animal = resolver_animal(registro)

    animal_existe = (
        cursor.execute("""
            SELECT 1
            FROM animais
            WHERE id_animal = ?
        """, (id_animal,)).fetchone()
        if id_animal else None
    )

    if not animal_existe:

        marcar_processado(
            registro,
            "NAO_VINCULADO",
            id_animal,
            "Animal não localizado."
        )

        continue

    diagnostico = texto(registro.get("diagnostico"))

    if not diagnostico:

        diagnostico = "Não informado"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "diagnosticos",
            id_animal,
            "diagnostico",
            registro.get("diagnostico"),
            diagnostico,
            "diagnostico_ausente",
            "Campo preenchido com indicação de ausência."
        )

    data_diagnostico = data_valida(
        registro.get("data_diagnostico")
    )

    if not data_diagnostico:

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "diagnosticos",
            id_animal,
            "data_diagnostico",
            registro.get("data_diagnostico"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    situacao = registro.get("situacao")

    if situacao not in SITUACOES_DIAGNOSTICO:
        situacao = "Em acompanhamento"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "diagnosticos",
            id_animal,
            "situacao",
            registro.get("situacao"),
            situacao,
            "situacao_invalida",
            "Situação padronizada."
        )

    inserir_dinamico(
        "diagnosticos",
        {
            "id_animal": id_animal,
            "data_diagnostico": data_diagnostico,
            "diagnostico": diagnostico,
            "descricao": texto(registro.get("descricao")),
            "situacao": situacao,
            "origem": "Sintético"
        }
    )

    marcar_processado(
        registro,
        "TRATADO",
        id_animal
    )

    quantidade_diagnosticos += 1


# 11. TRATAMENTO DE MAUS-TRATOS

quantidade_maus_tratos = 0

for registro in registros_por_conjunto["maus_tratos"]:

    id_bruto = registro["_id_registro"]

    if cursor.execute("""
        SELECT 1
        FROM controle_processamento
        WHERE id_registro_bruto = ?
    """, (id_bruto,)).fetchone():
        continue

    id_animal = resolver_animal(registro)

    if id_animal:

        animal_existe = cursor.execute("""
            SELECT 1
            FROM animais
            WHERE id_animal = ?
        """, (id_animal,)).fetchone()

        if not animal_existe:
            id_animal = None

    municipio = registro.get("municipio")

    if municipio not in MUNICIPIOS_VALIDOS:

        municipio = "Não informado"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "municipio",
            registro.get("municipio"),
            municipio,
            "municipio_invalido",
            "Município não recuperável com segurança."
        )

    tipo_ocorrencia = texto(
        registro.get("tipo_ocorrencia")
    )

    if not tipo_ocorrencia:

        tipo_ocorrencia = "Não informado"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "tipo_ocorrencia",
            registro.get("tipo_ocorrencia"),
            tipo_ocorrencia,
            "tipo_ocorrencia_ausente",
            "Campo preenchido com indicação de ausência."
        )

    data_ocorrencia = data_valida(
        registro.get("data_ocorrencia")
    )

    if not data_ocorrencia:

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "data_ocorrencia",
            registro.get("data_ocorrencia"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    situacao = registro.get("situacao")

    if situacao not in SITUACOES_OCORRENCIA:
        situacao = "Em investigação"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "situacao",
            registro.get("situacao"),
            situacao,
            "situacao_invalida",
            "Situação padronizada."
        )

    inserir_dinamico(
        "maus_tratos",
        {
            "id_animal": id_animal,
            "municipio": municipio,
            "data_ocorrencia": data_ocorrencia,
            "tipo_ocorrencia": tipo_ocorrencia,
            "descricao": texto(registro.get("descricao")),
            "situacao": situacao,
            "origem": "Sintético"
        }
    )

    marcar_processado(
        registro,
        "TRATADO",
        id_animal
    )

    quantidade_maus_tratos += 1


# 12. TRATAMENTO DE ÓBITOS E EUTANÁSIAS

quantidade_eventos = 0

for registro in registros_por_conjunto["obitos_eutanasias"]:

    id_bruto = registro["_id_registro"]

    if cursor.execute("""
        SELECT 1
        FROM controle_processamento
        WHERE id_registro_bruto = ?
    """, (id_bruto,)).fetchone():
        continue

    id_animal = resolver_animal(registro)

    if id_animal:

        animal_existe = cursor.execute("""
            SELECT 1
            FROM animais
            WHERE id_animal = ?
        """, (id_animal,)).fetchone()

        if not animal_existe:
            id_animal = None

    tipo_evento = registro.get("tipo_evento")

    if tipo_evento not in TIPOS_EVENTO:

        tipo_evento = "Não classificado"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "tipo_evento",
            registro.get("tipo_evento"),
            tipo_evento,
            "tipo_evento_invalido",
            "Evento mantido, mas encaminhado para classificação."
        )

    data_evento = data_valida(
        registro.get("data_evento")
    )

    motivo = texto(registro.get("motivo"))

    if not motivo:

        motivo = "Não informado"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "motivo",
            registro.get("motivo"),
            motivo,
            "motivo_ausente",
            "Campo preenchido com indicação de ausência."
        )

    municipio = registro.get("municipio")

    if municipio not in MUNICIPIOS_VALIDOS:
        municipio = "Não informado"

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "municipio",
            registro.get("municipio"),
            municipio,
            "municipio_invalido",
            "Município não recuperável com segurança."
        )

    # Se a data do evento for anterior ao acolhimento,
    # usa a data de acolhimento válida, quando disponível.
    if id_animal and data_evento:

        acolhimento = cursor.execute("""
            SELECT data_acolhimento
            FROM animais
            WHERE id_animal = ?
        """, (id_animal,)).fetchone()

        if (
            acolhimento
            and acolhimento["data_acolhimento"]
            and data_evento < acolhimento["data_acolhimento"]
        ):

            data_evento = acolhimento["data_acolhimento"]

            registrar_problema(
                id_bruto,
                registro["_id_lote"],
                "obitos_eutanasias",
                id_animal,
                "data_evento",
                registro.get("data_evento"),
                data_evento,
                "evento_anterior_ao_acolhimento",
                "Data ajustada para não anteceder o acolhimento."
            )

    if not data_evento:

        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "data_evento",
            registro.get("data_evento"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    inserir_dinamico(
        "obitos_eutanasias",
        {
            "id_animal": id_animal,
            "tipo_evento": tipo_evento,
            "data_evento": data_evento,
            "municipio": municipio,
            "motivo": motivo,
            "observacoes": texto(registro.get("observacoes")),
            "origem": "Sintético"
        }
    )

    marcar_processado(
        registro,
        "TRATADO",
        id_animal
    )

    quantidade_eventos += 1


# 13. ATUALIZAÇÃO DA SITUAÇÃO VACINAL DOS ANIMAIS

# Quando a situação original era inválida, usa os registros
# de vacinação como referência limitada.

# Não afirma que todas as vacinas estão em dia:
# apenas distingue existência de aplicação registrada.

for id_origem, id_final in mapa_animais.items():

    animal = cursor.execute("""
        SELECT situacao_vacinacao
        FROM animais
        WHERE id_animal = ?
    """, (id_final,)).fetchone()

    if not animal:
        continue

    if animal["situacao_vacinacao"] != "Não informado":
        continue

    aplicadas = cursor.execute("""
        SELECT COUNT(*)
        FROM vacinacoes
        WHERE id_animal = ?
          AND situacao = 'Aplicada'
    """, (id_final,)).fetchone()[0]

    if aplicadas > 0:
        nova_situacao = "Parcial"
    else:
        nova_situacao = "Pendente"

    cursor.execute("""
        UPDATE animais
        SET situacao_vacinacao = ?
        WHERE id_animal = ?
          AND origem = 'Sintético'
    """, (nova_situacao, id_final))


# 14. REGISTRO DE AUDITORIA

resumo = {
    "animais_inseridos": quantidade_animais_inseridos,
    "grupos_animais": len(grupos_animais),
    "duplicidades_identificadas": quantidade_duplicidades,
    "vacinacoes_inseridas": quantidade_vacinacoes,
    "diagnosticos_inseridos": quantidade_diagnosticos,
    "maus_tratos_inseridos": quantidade_maus_tratos,
    "eventos_inseridos": quantidade_eventos,
    "problemas_registrados": cursor.execute("""
        SELECT COUNT(*)
        FROM qualidade_dados
    """).fetchone()[0]
}

colunas_auditoria = colunas_tabela("auditoria")

dados_auditoria = {
    "entidade": "Tratamento de dados",
    "id_registro": None,
    "operacao": "TRATAMENTO",
    "descricao": json.dumps(
        resumo,
        ensure_ascii=False
    ),
    "data_operacao": datetime.now().isoformat(timespec="seconds")
}

inserir_dinamico(
    "auditoria",
    dados_auditoria
)


# 15. FINALIZAÇÃO

conexao.commit()

print("\n" + "=" * 60)
print("TRATAMENTO CONCLUÍDO")
print("=" * 60)

print("\nRESUMO DA EXECUÇÃO")

for campo, valor in resumo.items():

    print(
        f"{campo.replace('_', ' ').capitalize()}: {valor}"
    )

print("\nREGISTROS BRUTOS PRESERVADOS.")
print("CONTROLE DE PROCESSAMENTO ATUALIZADO.")
print("TABELA DE QUALIDADE ATUALIZADA.")

print("\nO tratamento pode ser executado novamente.")
print("Os registros já processados serão ignorados.")

conexao.close()


In [ ]:

import sqlite3
import json

# Reutiliza CAMINHO_BANCO definido na primeira celula.

conexao = sqlite3.connect(CAMINHO_BANCO)
conexao.row_factory = sqlite3.Row
cursor = conexao.cursor()

try:
    # Inicia uma transação para proteger o banco.
    cursor.execute("BEGIN IMMEDIATE")

    # 1. Carrega os IDs dos animais já existentes.
    ids_animais = {
        linha["id_animal"]
        for linha in cursor.execute("""
            SELECT id_animal
            FROM animais
        """).fetchall()
    }

    # 2. Recupera todos os registros brutos de animais.
    registros_brutos = cursor.execute("""
        SELECT id_registro, id_lote, dados_json
        FROM registros_brutos
        WHERE conjunto = 'animais'
        ORDER BY id_registro
    """).fetchall()

    # 3. Relaciona cada registro bruto ao animal definitivo.
    mapeamento = {}

    for registro in registros_brutos:
        dados = json.loads(registro["dados_json"])
        meta = dados.get("_meta") or {}

        id_origem = str(
            meta.get("id_animal_origem")
            or dados.get("id_animal")
            or ""
        ).strip()

        if not id_origem:
            raise Exception(
                f"Registro {registro['id_registro']} sem ID de origem."
            )

        if id_origem in ids_animais:
            id_final = id_origem

        elif "SINT-" + id_origem in ids_animais:
            id_final = "SINT-" + id_origem

        else:
            raise Exception(
                f"Animal sem correspondência: {id_origem}"
            )

        mapeamento[registro["id_registro"]] = {
            "id_lote": registro["id_lote"],
            "id_animal_final": id_final
        }

    # 4. Verifica controles de animais que já existem.
    controles_existentes = {
        linha["id_registro_bruto"]: linha
        for linha in cursor.execute("""
            SELECT id_registro_bruto, conjunto,
                   status, id_animal_final
            FROM controle_processamento
            WHERE conjunto = 'animais'
        """).fetchall()
    }

    for id_registro, dados in mapeamento.items():
        if id_registro in controles_existentes:
            existente = controles_existentes[id_registro]

            if (
                existente["status"] != "TRATADO"
                or existente["id_animal_final"]
                   != dados["id_animal_final"]
            ):
                raise Exception(
                    f"Controle existente inconsistente: {id_registro}"
                )

    # 5. Prepara somente os controles que estão faltando.
    novos_controles = []

    for id_registro, dados in mapeamento.items():

        if id_registro in controles_existentes:
            continue

        novos_controles.append((
            id_registro,
            dados["id_lote"],
            "animais",
            "1.0",
            "TRATADO",
            dados["id_animal_final"],
            "Animal consolidado na tabela animais. "
            "Controle recuperado após correção da rotina."
        ))

    # 6. Insere os controles faltantes.
    cursor.executemany("""
        INSERT OR IGNORE INTO controle_processamento (
            id_registro_bruto,
            id_lote,
            conjunto,
            versao_tratamento,
            status,
            id_animal_final,
            observacao,
            data_processamento
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, datetime('now'))
    """, novos_controles)

    # 7. Confere se todos os registros foram controlados.
    controles_finais = {
        linha["id_registro_bruto"]: linha
        for linha in cursor.execute("""
            SELECT id_registro_bruto, status, id_animal_final
            FROM controle_processamento
            WHERE conjunto = 'animais'
        """).fetchall()
    }

    for id_registro, dados in mapeamento.items():
        controle = controles_finais.get(id_registro)

        if (
            controle is None
            or controle["status"] != "TRATADO"
            or controle["id_animal_final"]
               != dados["id_animal_final"]
        ):
            raise Exception(
                f"Falha na verificação do registro {id_registro}."
            )

    if len(controles_finais) != len(registros_brutos):
        raise Exception(
            "A quantidade de controles não corresponde "
            "à quantidade de registros brutos."
        )

    # 8. Confirma a operação somente após as verificações.
    conexao.commit()

    print("=" * 55)
    print("RECUPERAÇÃO DO CONTROLE CONCLUÍDA")
    print("=" * 55)

    print("Registros brutos de animais:", len(registros_brutos))
    print("Controles já existentes e válidos:", len(controles_existentes))
    print("Novos controles inseridos:", len(novos_controles))
    print("Total de controles de animais:", len(controles_finais))

    print("\nCONTROLES POR CONJUNTO E STATUS")
    print("-" * 55)

    resumo = cursor.execute("""
        SELECT conjunto, status, COUNT(*) AS quantidade
        FROM controle_processamento
        GROUP BY conjunto, status
        ORDER BY conjunto, status
    """).fetchall()

    for linha in resumo:
        print(
            f"{linha['conjunto']}: "
            f"{linha['status']} = {linha['quantidade']}"
        )

    total = cursor.execute("""
        SELECT COUNT(*) AS total
        FROM controle_processamento
    """).fetchone()["total"]

    print("\nTotal geral de controles:", total)

except Exception as erro:
    conexao.rollback()

    print("ERRO: a operação foi cancelada.")
    print("Nenhuma alteração desta transação foi confirmada.")
    print("Detalhe:", erro)

finally:
    conexao.close()


In [ ]:
import sqlite3
import json

# Reutiliza CAMINHO_BANCO definido na primeira celula.

conexao = sqlite3.connect(CAMINHO_BANCO)
conexao.row_factory = sqlite3.Row
cursor = conexao.cursor()

# Recupera os IDs existentes na tabela definitiva
ids_animais = {
    linha["id_animal"]
    for linha in cursor.execute(
        "SELECT id_animal FROM animais"
    ).fetchall()
}

# Recupera os registros brutos de animais
brutos = cursor.execute("""
    SELECT id_registro, id_lote, dados_json
    FROM registros_brutos
    WHERE conjunto = 'animais'
    ORDER BY id_registro
""").fetchall()

encontrados = 0
nao_encontrados = []

for registro in brutos:
    dados = json.loads(registro["dados_json"])
    meta = dados.get("_meta") or {}

    id_origem = str(
        meta.get("id_animal_origem")
        or dados.get("id_animal")
        or ""
    ).strip()

    if id_origem in ids_animais:
        encontrados += 1
    elif "SINT-" + id_origem in ids_animais:
        encontrados += 1
    else:
        nao_encontrados.append(
            (registro["id_registro"], id_origem)
        )

print("VERIFICAÇÃO DOS ANIMAIS")
print("=" * 40)
print("Registros brutos:", len(brutos))
print("Animais correspondentes:", encontrados)
print("Sem correspondência:", len(nao_encontrados))

if nao_encontrados:
    print("\nPrimeiros registros sem correspondência:")
    for item in nao_encontrados[:10]:
        print(item)

conexao.close()